[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [SQLModel, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)

# table=True &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's Setup, with the `Hero` class its worked examples wrote and the
`deadpond` it validated. Run it first. Task 5 empties `SQLModel.metadata`, so the tasks run in the
order they are written.


In [1]:
import contextlib
import re
import subprocess
import sys
import warnings
from importlib.metadata import PackageNotFoundError, version

try:
    version("sqlmodel")
except PackageNotFoundError:                                        # Colab has no SQLModel: install the pinned version
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "sqlmodel==0.0.42"], check=True)

import pydantic
import sqlalchemy
import sqlmodel
from pydantic import BaseModel, ValidationError
from sqlalchemy.dialects import sqlite
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column
from sqlalchemy.schema import CreateTable
from sqlmodel import Field, SQLModel

TEAMS = [                                                           # name, headquarters
    ("Preventers", "Sharp Tower"),
    ("Z-Force", "Sister Margaret's Bar"),
    ("Wakaland Guard", "Grand Palace"),                             # no heroes, for the joins that keep a team anyway
]

HEROES = [                                                          # name, secret name, age, team
    ("Deadpond", "Dive Wilson", None, "Z-Force"),
    ("Spider-Boy", "Pedro Parqueador", 16, "Preventers"),
    ("Rusty-Man", "Tommy Sharp", 48, "Preventers"),
    ("Tarantula", "Natalia Roman-on", 32, "Preventers"),
    ("Black Lion", "Trevor Challa", 35, "Z-Force"),
    ("Dr. Weird", "Steve Weird", 36, "Z-Force"),
    ("Captain North America", "Esteban Rogelios", 93, "Preventers"),
    ("Princess Sure-E", "Sure-E", None, None),                      # on no team
]


def message(error):
    """An error's text, without the memory address or the version link that make no two runs agree."""
    text = re.sub(r"0x[0-9a-f]+", "0x...", str(error))
    return "\n".join(line for line in text.splitlines() if "errors.pydantic.dev" not in line).strip()


def fields(model):
    """A model's values in the order its class declares them, which a loaded object does not keep."""
    return {name: getattr(model, name) for name in type(model).model_fields}


@contextlib.contextmanager
def catching():
    """Collects warnings instead of printing them: Python prints one with the file that raised it."""
    with warnings.catch_warnings(record=True) as raised:
        warnings.simplefilter("always")
        yield raised


def table_sql(model):
    """The CREATE TABLE a table model describes, written for SQLite with no database anywhere."""
    return str(CreateTable(model.__table__).compile(dialect=sqlite.dialect())).strip()


print("sqlmodel", sqlmodel.__version__, "| sqlalchemy", sqlalchemy.__version__, "| pydantic", pydantic.VERSION)


class Hero(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    secret_name: str = Field(max_length=60)
    age: int | None = Field(default=None, index=True)


deadpond = Hero.model_validate({"name": "Deadpond", "secret_name": "Dive Wilson", "age": "30"})


sqlmodel 0.0.42 | sqlalchemy 2.0.54 | pydantic 2.12.5


**1.** A team, and the table it describes.


In [2]:
class Team(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(index=True, max_length=50)
    headquarters: str = Field(max_length=60)


print(table_sql(Team))
print("indexes:", sorted(index.name for index in Team.__table__.indexes))


CREATE TABLE team (
	id INTEGER NOT NULL, 
	name VARCHAR(50) NOT NULL, 
	headquarters VARCHAR(60) NOT NULL, 
	PRIMARY KEY (id)
)
indexes: ['ix_team_name']


The same three parts on `id` as on a hero's, and `index=True` on the name, which is what a lookup by
name reads.


**2.** What accepts a null, and what the model requires.


In [3]:
print("nullable columns:", [column.name for column in Hero.__table__.columns if column.nullable])
print("required fields :", Hero.model_json_schema()["required"])
# id is the primary key, and a primary key is never null, so its column is NOT NULL. The model does
# not require it either, since it defaults to None until the database fills it in.


nullable columns: ['age']
required fields : ['name', 'secret_name']


`age` is the one column that accepts a null, and it is a value a hero may not have. `id` is in
neither list: as a primary key its column is `NOT NULL`, and the model does not ask for it because
the database supplies it.


**3.** An age that arrives as text, twice.


In [4]:
validated = Hero.model_validate({"name": "Rusty-Man", "secret_name": "Tommy Sharp", "age": "48"})
print("model_validate:", repr(validated.age), type(validated.age).__name__)

built = Hero(name="Rusty-Man", secret_name="Tommy Sharp", age="48")
print("the constructor:", repr(built.age), type(built.age).__name__)


model_validate: 48 int
the constructor: '48' str


`model_validate` converted the text into the number the annotation asks for. The constructor did not
convert and did not refuse, and the hero now carries a string in a column declared `INTEGER`. The
**Validation and table=True** notebook is where that goes.

**4.** The model a service would receive.


In [5]:
class HeroCreate(SQLModel):                                         # no table=True
    name: str
    secret_name: str
    age: int | None = None


print("has a table:", hasattr(HeroCreate, "__table__"), "| tables:", sorted(SQLModel.metadata.tables))
try:
    HeroCreate(name="Mystery Man", secret_name="Unknown", age="old")
except ValidationError as error:
    print(message(error))


has a table: False | tables: ['hero', 'team']
1 validation error for HeroCreate
age
  Input should be a valid integer, unable to parse string as an integer [type=int_parsing, input_value='old', input_type=str]


No `id`, since the client does not choose one, and no table, so `SQLModel.metadata` still holds only
`hero` and the `team` from task 1. Its constructor validates, which is what a model that takes data
from outside is for.


**5.** The same class twice, with the metadata emptied in between.


In [6]:
class Sidekick(SQLModel, table=True):
    id: int | None = Field(default=None, primary_key=True)
    name: str = Field(max_length=40)
    catchphrase: str = Field(max_length=40)


print(table_sql(Sidekick))
print("indexes:", sorted(index.name for index in Sidekick.__table__.indexes))

with catching() as warned:
    SQLModel.metadata.clear()

    class Sidekick(SQLModel, table=True):
        id: int | None = Field(default=None, primary_key=True)
        name: str = Field(index=True, max_length=40)
        catchphrase: str = Field(max_length=40)


print("indexes now:", sorted(index.name for index in Sidekick.__table__.indexes))
print("tables now:", sorted(SQLModel.metadata.tables))
print("warned:", [str(warning.message).split(",")[0] for warning in warned])


CREATE TABLE sidekick (
	id INTEGER NOT NULL, 
	name VARCHAR(40) NOT NULL, 
	catchphrase VARCHAR(40) NOT NULL, 
	PRIMARY KEY (id)
)
indexes: []
indexes now: ['ix_sidekick_name']
tables now: ['sidekick']
warned: ['This declarative base already contains a class with the same class name and module name as __main__.Sidekick']


The `CREATE TABLE` does not change, since an index is a statement of its own, and `ix_sidekick_name`
appears in the table's indexes. `clear()` took `hero`, `team` and the first `sidekick` with it,
which is why the second definition is the only table left, and SQLAlchemy noted that the class name
is being used again.


**6.** A public hero, with no secret name in it.


In [7]:
class HeroPublic(SQLModel):
    id: int
    name: str
    age: int | None = None


def to_public(hero):
    """The fields of a hero that a response may carry."""
    return HeroPublic(id=hero.id, name=hero.name, age=hero.age)


deadpond.id = 1                                                     # the database has not run yet
public = to_public(deadpond)
print(fields(public))
print("fields of the public model:", list(HeroPublic.model_fields))


{'id': 1, 'name': 'Deadpond', 'age': 30}
fields of the public model: ['id', 'name', 'age']


The secret name is not in `HeroPublic`, so it cannot reach a response by accident. Writing the
conversion by hand is fine for three fields; the **Create, Read and Update Models** notebook builds
the family properly, with the shared fields in one place.


---

&#8592; **Back to:** [table=True](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/sqlmodel-deep-dive/01-table-true.ipynb)  &nbsp;&middot;&nbsp;  [SQLModel, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/sqlmodel-deep-dive.html)
